# ICHAN-DH B1 Presentation Walkthrough

This notebook demonstrates the prepared canonical array interface. It is a presentation artifact, not the complete Gate 5 verification notebook. It is delivered without execution outputs or experimental claims. Use a fresh kernel and start Jupyter from the repository root, the `presentation` directory, or this notebook directory.

## 1. Scope

The walkthrough uses a controlled unsigned 12-bit stored-value array. It shows deterministic strata, the 17-byte header, the variable safe prefix, guarded capacity, map-free payload extraction, and descriptive distortion metrics.

In [ ]:
%pip install "numpy>=1.26,<3" "scikit-image>=0.24,<0.27" "pydicom==3.0.2" matplotlib

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

if any(name == "ichan_secure" or name.startswith("ichan_secure.") for name in sys.modules):
    raise RuntimeError("Use a fresh kernel before running this presentation notebook")

source_candidates = [
    Path.cwd() / "assets/source",
    Path.cwd() / "ichansecure/assets/source",
    Path.cwd() / "presentation/ichansecure/assets/source",
]
source_matches = [path.resolve() for path in source_candidates if (path / "ichan_secure/__init__.py").is_file()]
if len(source_matches) != 1:
    raise RuntimeError(f"Expected exactly one ICHAN-DH source tree, found: {source_matches}")
SOURCE_ROOT = source_matches[0]
sys.path.insert(0, str(SOURCE_ROOT))

import ichan_secure
module_path = Path(ichan_secure.__file__).resolve()
if SOURCE_ROOT not in module_path.parents:
    raise RuntimeError(f"Unexpected ichan_secure import: {module_path}")

from ichan_secure import (
    Domain, PaddingSpec, build_header, capacity, classify, embed, extract,
    header_schedule, parse_header, select_profile,
)
from ichan_secure.metrics import distortion_metrics, recovery_metrics

## 2. Stored-value domain and controlled fixture

For unsigned 12-bit data, the declared domain is 0 to 4095. The fixture spans the complete domain once and contains no declared padding. It is not clinical data.

In [ ]:
domain = Domain(bits_stored=12, pixel_representation=0)
padding = PaddingSpec()
cover = np.arange(4096, dtype=np.uint16).reshape(64, 64)
payload = b"ICHAN-DH controlled presentation payload"
labels = classify(cover, domain)

{
    "domain_minimum": domain.minimum,
    "domain_maximum": domain.maximum,
    "class_counts": tuple(int(np.count_nonzero(labels == k)) for k in range(3)),
    "payload_bytes": len(payload),
}

## 3. Header and safe carrier schedule

The header stores `ICHS`, version 1, an unsigned 64-bit payload byte length, and a payload CRC32. The first 136 eligible one-bit carriers define the header positions and the reserved prefix endpoint $\beta$. CRC32 is not authentication.

In [ ]:
header_bytes = build_header(payload)
header = parse_header(header_bytes)
schedule = header_schedule(cover, domain, padding)

{
    "header_hex": header_bytes.hex(),
    "header_bytes": len(header_bytes),
    "declared_payload_bytes": header.payload_length,
    "checksum_hex": f"{header.checksum:08x}",
    "header_carriers": len(schedule.positions),
    "beta": schedule.beta,
}

## 4. Guarded capacity and profile selection

Capacity is reported as raw bits, complete payload bytes, usable bits, and tail bits. The smallest byte-feasible profile is selected.

In [ ]:
report = capacity(cover, domain, padding)
profile = select_profile(report, len(payload))

{
    "beta": report.beta,
    "raw_bits": report.raw_bits,
    "max_payload_bytes": report.max_payload_bytes,
    "usable_bits": report.usable_bits,
    "tail_bits": report.tail_bits,
    "selected_profile": profile,
}

## 5. Embedding and extraction

The sender returns only the stego array. The receiver reconstructs the schedule and profile from the received array and identical domain and padding metadata. Exact payload recovery must be observed after execution.

In [ ]:
stego = embed(cover, payload, domain, padding)
recovered = extract(stego, domain, padding)
recovery = recovery_metrics(payload, recovered)
distortion = distortion_metrics(cover, stego, domain)

{
    "recovery": asdict(recovery),
    "distortion": asdict(distortion),
    "payload_preview": recovered[:64],
}

## 6. Technical visualization

The visualization distinguishes the stored-value distribution, deterministic strata, and modifications. It does not represent Hounsfield units, display-window appearance, or diagnostic preservation.

In [ ]:
difference = stego.astype(np.int32) - cover.astype(np.int32)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(cover.ravel(), bins=64, color="#355c7d")
axes[0].set_title("Stored-value distribution")
axes[0].set_xlabel("Stored value")
axes[0].set_ylabel("Count")
label_image = axes[1].imshow(labels, cmap="viridis", vmin=0, vmax=2)
axes[1].set_title("Deterministic strata")
axes[1].axis("off")
difference_image = axes[2].imshow(difference, cmap="coolwarm", vmin=-7, vmax=7)
axes[2].set_title("Stego minus cover")
axes[2].axis("off")
fig.colorbar(label_image, ax=axes[1], ticks=[0, 1, 2], fraction=0.046, pad=0.04)
fig.colorbar(difference_image, ax=axes[2], fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

## 7. Restricted CT DICOM template

Set `RUN_DICOM` to `True` only after selecting an accepted, non-sensitive CT fixture and reviewing the file-interface policy. The derived output must use a new path.

In [ ]:
RUN_DICOM = False
SOURCE_DICOM = Path("data/replace-with-approved-ct-input.dcm")
STEGO_DICOM = Path("outputs/ichan-derived-stego.dcm")

if RUN_DICOM:
    from ichan_secure.dicom_io import embed_dicom, extract_dicom, load_dicom

    STEGO_DICOM.parent.mkdir(parents=True, exist_ok=True)
    loaded = load_dicom(SOURCE_DICOM)
    written_path = embed_dicom(SOURCE_DICOM, STEGO_DICOM, payload, overwrite=False)
    recovered_from_dicom = extract_dicom(written_path)
    print({
        "source_shape": loaded.array.shape,
        "bits_stored": loaded.domain.bits_stored,
        "pixel_representation": loaded.domain.pixel_representation,
        "output_path": str(written_path),
        "exact_payload": recovered_from_dicom == payload,
    })

## 8. Interpretation boundary

A successful controlled run would support the tested internal behavior only. It would not establish original-cover recovery, encryption, authentication, robustness, steganalysis resistance, clinical safety, complete DICOM interoperability, or empirical superiority.